<a href="https://colab.research.google.com/github/FranBlanchard/Estadistica-Descriptiva/blob/main/Analisis_matriculas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Análisis Descriptivo de matrículas de la educación Superior en la Región del Maule-Chile (2021)

Dataset: **091_MATRICULAS_ED_SUPERIOR_MAULE_2021.xlsx**

Variables disponibles:
- **ID**: identificador del estudiante
- **EDAD**: edad del estudiante matriculado
- **COMUNA SEDE**: comuna donde se ubica la sede en la que está matriculado

Se aplican técnicas de **estadística descriptiva** y visualizaciones con **Plotly** para explorar la distribución de edades y la concentración de matrículas por comuna.

**IMPORTAR LIBRERIAS**

In [32]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

**CARGAR LOS DATOS**

In [33]:
df = pd.read_excel('091_MATRICULAS_ED_SUPERIOR_MAULE_2021.xlsx', sheet_name='BASE DE DATOS')
edad = df['EDAD']
n = len(edad)

**Tabla de distribución de frecuencias (regla de Sturges)**

In [46]:
k = int(np.ceil(1 + np.log2(n)))                 # n° de clases
bins = np.histogram_bin_edges(edad, bins=k)
frec_abs, _ = np.histogram(edad, bins=bins)

tabla = pd.DataFrame({
    'intervalo': [f'[{bins[i]:.0f} - {bins[i+1]:.0f}) años' for i in range(len(bins) - 1)],
    'marca_clase': [(bins[i] + bins[i+1]) / 2 for i in range(len(bins) - 1)],
    'frecuencia_abs': frec_abs,
})
tabla['frecuencia_rel_%'] = (tabla['frecuencia_abs'] / n * 100).round(2)
tabla['frec_acumulada'] = tabla['frecuencia_abs'].cumsum()
tabla['frec_acumulada_%'] = tabla['frecuencia_rel_%'].cumsum().round(2)

print(tabla)

         intervalo  marca_clase  frecuencia_abs  frecuencia_rel_%  \
0   [18 - 21) años    19.705882           18589             38.35   
1   [21 - 25) años    23.117647           15844             32.69   
2   [25 - 28) años    26.529412            6097             12.58   
3   [28 - 32) años    29.941176            2608              5.38   
4   [32 - 35) años    33.352941            2187              4.51   
5   [35 - 38) años    36.764706            1030              2.13   
6   [38 - 42) años    40.176471             739              1.52   
7   [42 - 45) años    43.588235             626              1.29   
8   [45 - 49) años    47.000000             327              0.67   
9   [49 - 52) años    50.411765             230              0.47   
10  [52 - 56) años    53.823529              88              0.18   
11  [56 - 59) años    57.235294              58              0.12   
12  [59 - 62) años    60.647059              30              0.06   
13  [62 - 66) años    64.058824   

In [47]:
tabla_torta = tabla.copy()
tabla_torta['grupo'] = tabla_torta['intervalo'].where(
    tabla_torta['marca_clase'] < 55, '55+ años'
)
tabla_agrupada = tabla_torta.groupby('grupo', as_index=False)['frecuencia_abs'].sum()

fig = px.pie(
    tabla_agrupada, names='grupo', values='frecuencia_abs',
    title='Alumnos matriculados por rango de edad',
    hole=0.4,
)
fig.update_traces(textinfo='percent+label')
fig.update_layout(template='plotly_white', width=800, height=600)

fig.show(renderer="colab")

**Histograma de densidad + curva empírica (KDE) vs. curva normal teórica**

In [48]:
x_grid = np.linspace(edad.min(), edad.max(), 400)
kde = stats.gaussian_kde(edad)
densidad_empirica = kde(x_grid)
densidad_normal = stats.norm.pdf(x_grid, loc=edad.mean(), scale=edad.std())

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=edad, histnorm='probability density', nbinsx=k,
    name='Datos (histograma)', marker_color='#4C78A8', opacity=0.55,
))
fig.add_trace(go.Scatter(
    x=x_grid, y=densidad_empirica, mode='lines', name='Curva empírica (KDE)',
    line=dict(color='#54A24B', width=3),
))
fig.add_trace(go.Scatter(
    x=x_grid, y=densidad_normal, mode='lines', name='Curva normal teórica',
    line=dict(color='#E45756', width=3, dash='dash'),
))
fig.add_vline(
    x=edad.mean(), line_dash='dot', line_color='#333333',
    annotation_text=f'Media = {edad.mean():.1f}', annotation_position='top',
)
fig.update_layout(
    title='Histograma de densidad: curva empírica vs. curva normal teórica',
    xaxis_title='Edad (años)', yaxis_title='Densidad',
    template='plotly_white', width=900, height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
)

fig.show(renderer="colab")  # en Colab esto se ve interactivo sin necesitar kaleido


**Concentración fuerte en edades jóvenes:**

Los dos primeros intervalos (18–21.4 y 21.4–24.8 años) juntan cerca del 71% de toda la matrícula (38.35% + 32.69%). Es decir, 7 de cada 10 estudiantes matriculados tienen entre 18 y 25 años aproximadamente, lo cual es el perfil “tradicional” de ingreso a educación superior.

**Caída rápida y sostenida de la frecuencia:**

A partir del tercer intervalo (24.8–28.2 años, 12.58%) la frecuencia baja fuerte y sigue bajando intervalo tras intervalo, sin repuntes. Eso es justamente lo que define una distribución asimétrica a la derecha: no hay un segundo grupo grande en edades mayores, solo una cola larga y delgada.


**La cola es muy larga pero numéricamente insignificante:**

desde los ~35 años en adelante, cada intervalo aporta menos del 2.2% del total, y los últimos intervalos (65 a 76 años) tienen frecuencias de 1 a 7 personas. Son casos reales, pero estadísticamente son los valores atípicos (outliers)


**La frecuencia acumulada confirma la asimetría:**

al llegar a los ~29 años (fin del cuarto intervalo) ya se acumula el 89% de los datos. Eso significa que solo el 11% restante de los matriculados se reparte entre los 29 y 76 años, un rango de edad 6 veces más amplio que el que concentra al 89% de los casos.

**En resumen:**

la matrícula de educación superior en el Maule (2021) es mayoritariamente joven y homogénea en edad, con una minoría creciente pero pequeña de estudiantes “no tradicionales” (mayores de 30), y una cola de casos muy aislados (mayores de 50) que no representan un patrón, sino excepciones individuales.